# P111 — Deuda técnica oculta en los sistemas de aprendizaje automático

## 1. Título y paper

**Paper:** *Hidden Technical Debt in Machine Learning Systems*  
**Autoría:** D. Sculley, Gary Holt, Daniel Golovin, Eugene Davydov, Todd Phillips, Dietmar Ebner, Vinay Chaudhary, Michael Young, Jean-François Crespo, Dan Dennison  
**Año y venue:** 2015 · NeurIPS 2015  
**Nivel:** L1 · **Motor:** `deuda_tecnica`  
**Ficha completa:** [`P111_deuda_tecnica`](../../papers/foundational/P111_deuda_tecnica/README.md)

**Hito:** Nombra el hecho incómodo del área: el código del modelo es una fracción diminuta del sistema, y el resto acumula una deuda que ninguna herramienta detecta.

- [NeurIPS 2015](https://papers.nips.cc/paper_files/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los equipos medían su trabajo por la calidad del modelo mientras el sistema alrededor —ingestión, características, servicio, monitorización, configuración— crecía sin control. Y esa parte acumula formas de deuda que no tienen equivalente en software convencional: dependencias de datos que ningún compilador comprueba.
2. Ejecutar una implementación mínima de la propuesta: Un catálogo de antipatrones específicos del aprendizaje automático: dependencias de datos no declaradas, características huérfanas, bucles de realimentación ocultos, código de pegamento, deuda de configuración, y el principio CACE — cambiar cualquier cosa lo cambia todo.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P80


## 4. Intuición

La figura más citada del artículo es una caja pequeña que pone «ML code» rodeada de cajas enormes. El modelo es la parte diminuta. Todo lo demás —datos, características, servicio, monitorización, configuración, pegamento— es el sistema, y acumula una deuda que ninguna herramienta detecta.


## 5. Concepto mínimo

```text
Deuda específica del aprendizaje automático:
  · dependencias de DATOS que ningún compilador comprueba
  · características huérfanas que se calculan y nadie usa
  · bucles de realimentación ocultos
  · código de pegamento entre sistemas

CACE: Changing Anything Changes Everything
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('deuda_tecnica', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué porcentaje del sistema es el código del modelo?
2. ¿A cuántos consumidores afecta retirar una característica?
3. ¿Qué pasa aguas abajo si se cambia el umbral de un modelo?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('deuda_tecnica', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('deuda_tecnica', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El código del modelo son **800 líneas de 19 800**: el **4 %** del sistema. Retirar la característica «f_precio» afecta a **3 consumidores** que nadie tenía apuntados, hay **1 característica huérfana** que se sigue calculando sin que nadie la use, y bajar el umbral del modelo A de 0,5 a 0,45 hace que el modelo B reciba un **36 % más de entradas** sin que nadie lo haya tocado.


## 10. Comentario pedagógico

Las dependencias de datos son deuda invisible porque **no hay compilador que avise**. En software convencional, quitar una función rompe la compilación; quitar una característica no rompe nada hasta que un modelo empieza a predecir peor, semanas después y sin relación aparente con el cambio.


## 11. Error o anti-patrón deliberado

Anti-patrón: medir el progreso del equipo por la calidad del modelo.


In [ ]:
print('El modelo es el 4% del codigo y el 90% de la atencion del equipo.')
print('La deuda se acumula en el 96% restante, donde nadie mira.')
print('Y no se paga hasta que el sistema falla de una forma que nadie sabe explicar.')

## 12. Corrección

Dónde está la deuda y cómo se hace visible:


In [ ]:
r = run_paper_lab('deuda_tecnica', seed=7)['result']
for c in r['componentes']:
    print(f"  {c['parte']:<38} {c['lineas']:>6} lineas  {c['porcentaje']:>5} %")
print()
print('retirar', r['caracteristica_a_retirar'], '->', r['consumidores_afectados'])
print('huerfanas:', r['caracteristicas_huerfanas'])

## 13. Desafío guiado

Explica el principio CACE con el ejemplo del umbral, y por qué eso hace difícil razonar sobre una cadena de modelos.


In [ ]:
r = run_paper_lab('deuda_tecnica', seed=3)['result']
show(r)

## 14. Desafío autónomo

Dibuja el grafo de dependencias de datos de un sistema tuyo: qué característica alimenta a qué modelo y a qué informe. Después busca las huérfanas.


## 15. Evidencia de aprendizaje

Guarda el desglose por componente y tu grafo de dependencias con las características huérfanas identificadas.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P111_deuda_tecnica/README.md) · evaluación formal: [`assessments/papers/P111_deuda_tecnica.md`](../../assessments/papers/P111_deuda_tecnica.md)


## 16. Cierre

Ya está nombrada la deuda. La pregunta siguiente es cómo saber, antes de promocionar, si un sistema está listo.


## 17. Conexión con el siguiente hito

- P112
- P115

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
